# 03 · Feature Engineering
**FTTH Rollout Optimizer — WBS Capstone**

Stack: **Pandas · GeoPandas · OSMnx · Shapely · LlamaIndex**

Goals:
- Spatial features via GeoPandas + Shapely (proximity, density, clustering)
- OSMnx network features (road graph, walkability)
- LlamaIndex: index the feature table so we can query it in natural language
- Output: `data/processed/gemeinden_features.csv`

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, MultiPoint
from shapely.ops import nearest_points
import osmnx as ox
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROCESSED = Path('../data/processed')
df = pd.read_csv(PROCESSED / 'gemeinden_clean.csv')

# Rebuild GeoDataFrame from lat/lon
geometry = [Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
gdf_utm = gdf.to_crs('EPSG:25832')  # UTM 32N — metres, needed for distance calcs

print(f'Loaded: {gdf.shape[0]} Gemeinden')
print(f'CRS WGS84 : {gdf.crs}')
print(f'CRS UTM32N: {gdf_utm.crs}')

## 1 · Spatial Features — GeoPandas + Shapely

### 1.1 · Nearest-neighbour distance between Gemeinden
Proxy for network spill-over effect: if your neighbour already has FTTH, your adoption tends to be higher.

In [ ]:
from shapely.ops import nearest_points

def nearest_neighbour_dist(gdf_utm):
    """For each Gemeinde, find distance to closest other Gemeinde (metres)."""
    coords = np.array([(geom.x, geom.y) for geom in gdf_utm.geometry])
    dists = []
    for i, pt in enumerate(gdf_utm.geometry):
        others = MultiPoint([g for j, g in enumerate(gdf_utm.geometry) if j != i])
        nearest = nearest_points(pt, others)[1]
        dists.append(pt.distance(nearest))
    return np.array(dists)

# For 500 points this takes ~10 sec — use vectorised KD-tree for large datasets
from scipy.spatial import cKDTree

coords_utm = np.array([(geom.x, geom.y) for geom in gdf_utm.geometry])
tree = cKDTree(coords_utm)
# k=2: first match is self (dist=0), second is nearest neighbour
dists, _ = tree.query(coords_utm, k=2)
gdf_utm['dist_to_nearest_gemeinde_m'] = dists[:, 1].round(0).astype(int)
gdf['dist_to_nearest_gemeinde_m']     = gdf_utm['dist_to_nearest_gemeinde_m'].values

print(f"Nearest neighbour dist — mean: {gdf['dist_to_nearest_gemeinde_m'].mean():.0f}m "
      f"| min: {gdf['dist_to_nearest_gemeinde_m'].min()}m "
      f"| max: {gdf['dist_to_nearest_gemeinde_m'].max()}m")

### 1.2 · Spatial Lag — mean adoption of neighbours
Classic spatial feature: weighted average of surrounding Gemeinden adoption rates within radius.

In [ ]:
RADIUS_M = 25_000  # 25 km radius

spatial_lag = []
for i, (x, y) in enumerate(coords_utm):
    # find all points within radius
    idx_within = tree.query_ball_point([x, y], r=RADIUS_M)
    idx_within = [j for j in idx_within if j != i]  # exclude self
    if idx_within:
        spatial_lag.append(gdf_utm.iloc[idx_within]['adoption_rate_pct'].mean())
    else:
        spatial_lag.append(gdf_utm.iloc[i]['adoption_rate_pct'])  # fallback: own value

gdf['spatial_lag_adoption'] = np.round(spatial_lag, 2)

print(f"Spatial lag (25km radius) — mean: {gdf['spatial_lag_adoption'].mean():.2f}% "
      f"| std: {gdf['spatial_lag_adoption'].std():.2f}%")

### 1.3 · Gemeinde Area & Shape Complexity (Shapely)
Simulated Voronoi polygons as Gemeinde boundaries — approximates real cadastral shapes.

In [ ]:
from shapely.ops import voronoi_diagram
from shapely.geometry import MultiPoint as ShapelyMP

# Build Voronoi diagram as proxy for Gemeinde polygons
all_points = ShapelyMP(list(gdf_utm.geometry))
regions    = voronoi_diagram(all_points)

# Match Voronoi regions back to points (spatial join)
voronoi_gdf = gpd.GeoDataFrame(geometry=list(regions.geoms), crs='EPSG:25832')
joined = gpd.sjoin(gdf_utm[['gemeinde_id','geometry']], voronoi_gdf, how='left', predicate='within')

# Area in km²
voronoi_gdf['area_km2'] = (voronoi_gdf.area / 1_000_000).round(2)

# Merge area back
area_map = {}
for _, row in joined.iterrows():
    if not pd.isna(row.get('index_right')):
        idx = int(row['index_right'])
        area_map[row['gemeinde_id']] = voronoi_gdf.iloc[idx]['area_km2']

gdf['area_km2'] = gdf['gemeinde_id'].map(area_map).fillna(gdf['pop_density_km2'].apply(lambda x: 1000/max(x,1))).round(2)

# Derived: population per km² recalculated from Voronoi area
gdf['pop_per_area'] = (gdf['population'] / gdf['area_km2'].clip(lower=0.1)).round(1)

print(f"Area km² — mean: {gdf['area_km2'].mean():.1f} | median: {gdf['area_km2'].median():.1f}")

## 2 · OSMnx Network Features

For the full dataset we use **pre-computed synthetic proxies** (instant, no API needed).
The real OSMnx call is shown in the commented block — swap for production use.

Features extracted from road graph:
- `avg_node_degree` — connectivity of road network
- `street_density_km_per_km2` — total road length per area
- `avg_circuity` — ratio of road distance to straight-line distance (terrain proxy)

In [ ]:
# --- SYNTHETIC OSMnx proxies (runs instantly) ---
# Correlated with pop_density and terrain_slope for realism
np.random.seed(42)
n = len(gdf)

gdf['osm_avg_node_degree']        = (2.5 + 0.8 * (gdf['pop_density_km2'] / gdf['pop_density_km2'].max())
                                      + np.random.normal(0, 0.2, n)).clip(1.5, 4.5).round(2)
gdf['osm_street_density_km_km2']  = (gdf['road_length_km'] / gdf['area_km2'].clip(0.1)).round(2)
gdf['osm_avg_circuity']           = (1.05 + 0.015 * gdf['terrain_slope_deg']
                                      + np.random.normal(0, 0.02, n)).clip(1.0, 1.8).round(3)

print('OSMnx proxy features computed.')
print(gdf[['osm_avg_node_degree','osm_street_density_km_km2','osm_avg_circuity']].describe().round(2))

In [ ]:
# --- REAL OSMnx (uncomment for production — requires internet) ---
#
# import osmnx as ox
#
# place = 'Landsberg am Lech, Bavaria, Germany'
# G = ox.graph_from_place(place, network_type='drive')
#
# stats = ox.basic_stats(G)
# print(f"Node count     : {stats['n']}")
# print(f"Edge count     : {stats['m']}")
# print(f"Avg node degree: {stats['k_avg']:.2f}")
# print(f"Street density : {stats['street_density_km']:.2f} km/km²")
# print(f"Avg circuity   : {stats['circuity_avg']:.3f}")
#
# # For all Gemeinden, loop with ox.graph_from_point(lat, lon, dist=5000)
# # and extract stats per point — takes ~2-4h for 500 municipalities
# # Cache results to RAW / 'osmnx_stats.csv'

## 3 · Derived & Engineered Features

In [ ]:
# Coverage gap: how much room is there to grow?
gdf['coverage_gap_pct'] = (100 - gdf['existing_coverage_pct']).round(1)

# Infrastructure cost proxy (distance × terrain difficulty)
gdf['capex_difficulty_index'] = (
    (gdf['dist_to_pop_node_m'] / gdf['dist_to_pop_node_m'].max()) * 0.5 +
    (gdf['terrain_slope_deg']  / gdf['terrain_slope_deg'].max())  * 0.3 +
    (gdf['osm_avg_circuity'] - 1) / 0.8                          * 0.2
).round(3)

# Market attractiveness score
gdf['market_attractiveness'] = (
    (gdf['pop_density_km2']        / gdf['pop_density_km2'].max())        * 0.30 +
    (gdf['avg_household_income']   / gdf['avg_household_income'].max())   * 0.25 +
    (gdf['coverage_gap_pct']       / 100)                                 * 0.20 +
    (1 - gdf['competitor_coverage_pct'] / 80)                             * 0.15 +
    (gdf['spatial_lag_adoption']   / gdf['spatial_lag_adoption'].max())   * 0.10
).round(3)

# Log-transform skewed features (helps tree models too but critical for linear baselines)
for col in ['population', 'dist_to_pop_node_m', 'pop_density_km2', 'homes_passed']:
    gdf[f'log_{col}'] = np.log1p(gdf[col]).round(4)

# Income tier (ordinal)
gdf['income_tier'] = pd.cut(
    gdf['avg_household_income'],
    bins=[0, 30000, 40000, 50000, 999999],
    labels=[1, 2, 3, 4]
).astype(int)

print('Engineered features added:')
new_cols = ['coverage_gap_pct','capex_difficulty_index','market_attractiveness',
            'log_population','log_dist_to_pop_node_m','log_pop_density_km2',
            'log_homes_passed','income_tier']
print(gdf[new_cols].describe().round(3).T[['mean','std','min','max']])

## 4 · Final Feature Set

In [ ]:
FINAL_FEATURES = [
    # Demographic
    'log_population', 'pop_density_km2', 'log_pop_density_km2',
    'avg_household_income', 'income_tier',
    'share_over_65_pct', 'share_under_18_pct', 'unemployment_rate_pct',
    # Infrastructure
    'existing_coverage_pct', 'coverage_gap_pct',
    'dist_to_pop_node_m', 'log_dist_to_pop_node_m',
    'dist_to_cabinet_m', 'homes_passed', 'log_homes_passed',
    # Spatial
    'building_density', 'area_km2', 'pop_per_area',
    'dist_to_nearest_gemeinde_m', 'spatial_lag_adoption',
    # OSMnx
    'osm_avg_node_degree', 'osm_street_density_km_km2', 'osm_avg_circuity',
    # Terrain
    'terrain_slope_deg', 'road_length_km',
    # Competition
    'competitor_coverage_pct',
    # Composite
    'capex_difficulty_index', 'market_attractiveness',
]

TARGET = 'adoption_rate_pct'
ID_COLS = ['gemeinde_id', 'name', 'lat', 'lon']

df_final = gdf[ID_COLS + FINAL_FEATURES + [TARGET]].copy()
df_final = df_final.drop(columns='geometry', errors='ignore')

print(f'Final dataset: {df_final.shape[0]} rows × {df_final.shape[1]} columns')
print(f'Features: {len(FINAL_FEATURES)} | Target: {TARGET}')
df_final.head(3)

## 5 · LlamaIndex — Natural Language Query over Feature Data

Index the feature table so you (or a stakeholder) can ask questions like:
- *"Which Gemeinden have high market attractiveness but low existing coverage?"*
- *"What is the average CAPEX difficulty for municipalities with adoption > 60%?"*

In [ ]:
# LlamaIndex over the feature table
# Requires: pip install llama-index llama-index-llms-ollama (local) OR openai key

try:
    from llama_index.core import VectorStoreIndex, Document, Settings
    from llama_index.llms.ollama import Ollama
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding

    LLAMAINDEX_AVAILABLE = True
    print('LlamaIndex available — building index...')
except ImportError:
    LLAMAINDEX_AVAILABLE = False
    print('LlamaIndex not installed — skipping RAG section.')
    print('Install: pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface')

In [ ]:
if LLAMAINDEX_AVAILABLE:
    # Use local Ollama (llama3) + MiniLM embeddings — no API key needed
    Settings.llm   = Ollama(model='llama3', request_timeout=120.0)
    Settings.embed_model = HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2')

    # Convert top-100 Gemeinden to documents
    top_100 = df_final.nlargest(100, 'market_attractiveness')

    documents = []
    for _, row in top_100.iterrows():
        text = (
            f"Gemeinde: {row['name']} (ID {row['gemeinde_id']})\n"
            f"Location: lat={row['lat']:.3f}, lon={row['lon']:.3f}\n"
            f"Adoption rate: {row['adoption_rate_pct']:.1f}%\n"
            f"Market attractiveness: {row['market_attractiveness']:.3f}\n"
            f"CAPEX difficulty: {row['capex_difficulty_index']:.3f}\n"
            f"Population density: {row['pop_density_km2']:.0f} inh/km²\n"
            f"Avg household income: €{row['avg_household_income']:,.0f}\n"
            f"Coverage gap: {row['coverage_gap_pct']:.1f}%\n"
            f"Distance to POP node: {row['dist_to_pop_node_m']}m\n"
            f"Spatial lag adoption: {row['spatial_lag_adoption']:.1f}%\n"
            f"Competitor coverage: {row['competitor_coverage_pct']:.1f}%"
        )
        documents.append(Document(text=text, doc_id=str(row['gemeinde_id'])))

    index   = VectorStoreIndex.from_documents(documents)
    engine  = index.as_query_engine(similarity_top_k=5)

    print(f'Index built with {len(documents)} documents.')

    # Example queries
    queries = [
        "Which Gemeinden have the highest market attractiveness and a coverage gap above 40%?",
        "What is the typical CAPEX difficulty for municipalities with adoption rate above 60%?",
        "Which areas should be prioritized for Phase 1 rollout based on ROI potential?",
    ]

    for q in queries:
        print(f'\nQ: {q}')
        response = engine.query(q)
        print(f'A: {response}')

## 6 · Save Final Feature Dataset

In [ ]:
out_path = PROCESSED / 'gemeinden_features.csv'
df_final.to_csv(out_path, index=False)

# Also save GeoPackage with geometry for QGIS
df_geo = df_final.copy()
geometry = [Point(lon, lat) for lon, lat in zip(df_geo['lon'], df_geo['lat'])]
gdf_out  = gpd.GeoDataFrame(df_geo, geometry=geometry, crs='EPSG:4326')
gdf_out.to_file(PROCESSED / 'gemeinden_features.gpkg', driver='GPKG')

print(f'Saved → {out_path}')
print(f'Saved → {PROCESSED / "gemeinden_features.gpkg"}')
print(f'Shape : {df_final.shape}')
print(f'\nFinal feature list ({len(FINAL_FEATURES)} features):')
for f in FINAL_FEATURES:
    print(f'  · {f}')

---
## Feature Engineering Summary

| Category | Features | Tool |
|----------|----------|------|
| Spatial proximity | `dist_to_nearest_gemeinde_m`, `spatial_lag_adoption` | GeoPandas + SciPy KD-Tree |
| Boundary area | `area_km2`, `pop_per_area` | Shapely Voronoi |
| Road network | `osm_avg_node_degree`, `osm_street_density_km_km2`, `osm_avg_circuity` | OSMnx |
| Log transforms | `log_population`, `log_dist_to_pop_node_m`, etc. | Pandas / NumPy |
| Composite | `market_attractiveness`, `capex_difficulty_index` | Domain knowledge |
| RAG query layer | Natural language over top-100 Gemeinden | LlamaIndex + Ollama |

**Total features going into ML: 29**

**Next:** `04_ml_model.ipynb`